In [ ]:
import requests
import pandas as pd
import sqlite3

Задание 1 (1.1)

In [ ]:
def catch_hh_vacancies(query = "Разработчик", only_with_salary = True, target = 500, per_page = 100):
    headers = {"User-Agent": "hh-anomaly-script"}
    items = []
    page = 0
    while len(items) < target:
        params = {
            "text": query,
            "only_with_salary": str(only_with_salary).lower(),
            "per_page": per_page,
            "page": page
        }
        resp = requests.get("https://api.hh.ru/vacancies", params = params, headers = headers)
        if resp.status_code != 200:
            raise RuntimeError(f"hh API вернул {resp.status_code}: {resp.text}")
        data = resp.json()
        page_items = data.get("items", [])
        if not page_items:
            break
        items.extend(page_items)
        print(f"Выбранная страница {page}: {len(page_items)} items, total {len(items)}")
        page += 1
        if page > 500:
            break
    return items

In [ ]:
def items_to_dataframe(items):
    df = pd.json_normalize(items)
    keep = [
        "id",
        "name",
        "area.name",
        "salary.from",
        "salary.to",
        "salary.gross",
        "salary.currency",
        "employer.name"
    ]
    for col in keep:
        if col not in df.columns:
            df[col] = np.nan
    df = df[["id","name","area.name","salary.from","salary.to","salary.gross","salary.currency","employer.name"]]
    return df

In [ ]:
df = items_to_dataframe(catch_hh_vacancies())

Выбранная страница 0: 100 items, total 100
Выбранная страница 1: 100 items, total 200
Выбранная страница 2: 100 items, total 300
Выбранная страница 3: 100 items, total 400
Выбранная страница 4: 100 items, total 500


In [ ]:
df.head()

,id,name,area.name,salary.from,salary.to,salary.gross,salary.currency,employer.name
0,128785531,Junior back-end developer,Ташкент,4000000.0,8000000.0,True,UZS,OOO UZGPS
1,128709717,Java Backend Developer (Вакансия только для гр...,Ташкент,20000000.0,NaN,True,UZS,ГУ ADLIYA ORGANLARI VA MUASSASALARIDA AXBOROT-...
2,128721524,Python Developer,Гродно,60000.0,110000.0,False,RUR,Деханд Владислав Дмитриевич
3,128774161,Strong Junior NodeJS Developer,Ташкент,7000000.0,NaN,True,UZS,UNICAL
4,128747824,QA Engineer,Ташкент,1000.0,2500.0,True,USD,EVERTECH


(1.2)

In [ ]:
df.to_csv('df.csv', index= False)
df.to_json('df.json', index= False)

Задание 2

In [ ]:
df_empty = df.iloc[0:0]

In [ ]:
con1 = sqlite3.connect('db1.sqlite')
df_empty.to_sql('db1', con1, if_exists = 'replace', index = False)
con1.close()

In [ ]:
con2 = sqlite3.connect('db2.sqlite')
df_empty.to_sql('db2', con2, if_exists = 'replace', index = False)
con2.close()

Задание 3

In [ ]:
con1 = sqlite3.connect('db1.sqlite')
df_csv = pd.read_csv('df.csv')
df_csv.to_sql('db1', con1, if_exists= 'append', index = False)
con1.close()

In [ ]:
con2 = sqlite3.connect('db2.sqlite')
df_csv = pd.read_json('df.json')
df_csv.to_sql('db2', con2, if_exists= 'append', index = False)
con2.close()

Задание 4

In [ ]:
con3 = sqlite3.connect("db3.sqlite")
cursor = con3.cursor()
cursor.execute("""
CREATE TABLE IF NOT EXISTS cities (
    city_id INTEGER PRIMARY KEY,
    city TEXT UNIQUE
)
""")

In [ ]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS employers (
    employer_id INTEGER PRIMARY KEY,
    employer TEXT UNIQUE
)
""")

In [ ]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS vacancies (
    id INTEGER PRIMARY KEY,
    name TEXT,
    salary_from INTEGER,
    salary_to INTEGER,
    salary_gross INTEGER CHECK (salary_gross IN (0, 1)),
    salary_currency TEXT,
    city_id INTEGER,
    employer_id INTEGER,
    FOREIGN KEY (city_id) REFERENCES cities(city_id),
    FOREIGN KEY (employer_id) REFERENCES employers(employer_id)
)
""")
con3.commit()
con3.close()

In [ ]:
cities = df['area.name'].dropna().unique()
df_cities = pd.DataFrame({
    'city_id': range(1, len(cities) + 1),
    'city': cities
})

In [ ]:
employers = df['employer.name'].dropna().unique()
df_employers = pd.DataFrame({
    'employer_id': range(1, len(employers) + 1),
    'employer': employers
})

In [ ]:
city_map = dict(zip(df_cities['city'], df_cities['city_id']))
employer_map = dict(zip(df_employers['employer'], df_employers['employer_id']))
df['city_id'] = df['area.name'].map(city_map)
df['employer_id'] = df['employer.name'].map(employer_map)
df_vacancies = df[['id', 'name', 'city_id', 'salary.to', 'salary.from', 'salary.currency', 'employer_id']]
df_vacancies.columns =['id', 'name', 'city_id', 'salary_to', 'salary_from', 'salary_currency', 'employer_id']

In [ ]:
con3 = sqlite3.connect("db3.sqlite")
df_cities.to_sql("cities", con3, if_exists="append", index=False)
df_employers.to_sql("employers", con3, if_exists="append", index=False)
df_vacancies.to_sql("vacancies", con3, if_exists="append", index=False)

con3.close()


Задание 5 и 6

In [ ]:
df['salary.currency'].unique()

array(['UZS', 'RUR', 'USD', 'BYR', 'KZT', 'EUR'], dtype=object)

In [ ]:
con3 = sqlite3.connect('db3.sqlite')
cursor = con3.cursor()
cursor.execute("ALTER TABLE vacancies ADD COLUMN salary_clean REAL;")

In [ ]:
cursor.execute("""
UPDATE vacancies
SET salary_clean =
    (CASE
    WHEN salary_from IS NULL THEN salary_to
    WHEN salary_to IS NULL THEN salary_from
    ELSE (salary_from + salary_to) / 2
    END)
    * CASE salary_currency
    WHEN 'UZS' THEN 0.007
    WHEN 'RUR' THEN 1
    WHEN 'USD' THEN 90
    WHEN 'KZT' THEN 0.21
    WHEN 'BYR' THEN 0.035
    WHEN 'EUR' THEN 100
    ELSE 1
END
    * 0.87
""")
con3.commit()
con3.close()

Задание 7 и 8

In [ ]:
con3 = sqlite3.connect('db3.sqlite')
cursor = con3.cursor()
query = """
SELECT v.id,
v.name,
c.city,
e.employer,
v.salary_clean
FROM vacancies v
JOIN cities c
USING(city_id)
JOIN employers e
USING(employer_id)
WHERE c.city IN ('Москва', 'Ташкент') AND v.salary_clean IS NOT NULL
ORDER BY v.salary_clean DESC """
cursor.execute(query)
con3.close()

Задание 9

In [ ]:
con3 = sqlite3.connect('db3.sqlite')
cursor = con3.cursor()
query = """
SELECT id, name, city, employer, salary_clean
FROM (SELECT v.id AS id,
v.name AS name,
c.city AS city,
e.employer AS employer,
v.salary_clean AS salary_clean,
DENSE_RANK() OVER(PARTITION BY c.city ORDER BY v.salary_clean DESC) AS rank
FROM vacancies v
JOIN cities c
USING(city_id)
JOIN employers e
USING(employer_id)
WHERE v.salary_clean IS NOT NULL) AS t1
WHERE rank = 1 """
cursor.execute(query)
con3.close()

Задание 10

In [ ]:
con3 = sqlite3.connect('db3.sqlite')
cursor = con3.cursor()
query = """
SELECT id, name, city, employer,
CASE
WHEN salary_min_index IS NULL THEN salary_clean
END salary_min,
CASE
WHEN salary_max_index IS NULL THEN salary_clean
END salary_max
FROM (SELECT v.id AS id,
v.name AS name,
c.city AS city,
e.employer AS employer,
v.salary_clean AS salary_clean,
LAG(v.salary_clean) OVER(PARTITION BY c.city ORDER BY v.salary_clean) AS salary_min_index,
LEAD(v.salary_clean) OVER(PARTITION BY c.city ORDER BY v.salary_clean) AS salary_max_index
FROM vacancies v
JOIN cities c
USING(city_id)
JOIN employers e
USING(employer_id)
WHERE v.salary_clean IS NOT NULL) AS t1
WHERE salary_min_index IS NULL or salary_max_index IS NULL"""
cursor.execute(query)
con3.close()